# 04. OCR Experiments: Images, GIFs & Videos (`OcrExperimentAdapter`)

In [1]:
import json
from pathlib import Path

import pandas as pd

from meme_organizer.config import CYRILLIC_OCR_DICT_PATH, CYRILLIC_OCR_MODEL_PATH, MLFLOW_EXPERIMENT_OCR
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.extraction.ocr import OcrPreprocessor, RapidOcrExtractor
from meme_organizer.extraction.ocr.ocr_experiment_adapter import OcrExperimentAdapter
from meme_organizer.extraction.video import KeyframeSampler

ROOT = Path("..").resolve()
GOLD_PATH = ROOT / "data" / "gold" / "gold_dataset.jsonl"
SILVER_DIR = ROOT / "data" / "silver"

records = [json.loads(line) for line in GOLD_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
img_records = [r for r in records if r["media_type"] == "image" and r.get("transcript_or_text", "").strip()]
vid_records = [r for r in records if r["media_type"] != "image" and r.get("transcript_or_text", "").strip()]
print(f"Images: {len(img_records)} | Videos/GIFs: {len(vid_records)}")

rapid_cyr = RapidOcrExtractor(
    min_confidence=0.5,
    rec_model_path=str(ROOT / CYRILLIC_OCR_MODEL_PATH),
    rec_keys_path=str(ROOT / CYRILLIC_OCR_DICT_PATH),
)
results = []

2026/11/27 23:10:05 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Images: 124 | Videos/GIFs: 115


In [2]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_OCR) as exp:
    adapter = OcrExperimentAdapter(
        run_name="img_rapidocr_cyrillic_raw",
        extractor=rapid_cyr,
        preprocessor=OcrPreprocessor(mode="raw"),
        sampler=KeyframeSampler(strategy="uniform_3"),
        records=img_records,
        silver_dir=SILVER_DIR,
        rec_model_path=ROOT / CYRILLIC_OCR_MODEL_PATH,
        rec_keys_path=ROOT / CYRILLIC_OCR_DICT_PATH,
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "img_raw", **res["quality"], **res["performance"]})
print("Images:", res["quality"], res["performance"])

2026/11/27 23:10:05 INFO mlflow.tracking.fluent: Experiment with name 'meme-organizer-ocr' does not exist. Creating a new experiment.


🏃 View run img_rapidocr_cyrillic_raw at: http://localhost:5000/#/experiments/18/runs/9e2c02b1ef904405ae1f85b2c4a43e5c
🧪 View experiment at: http://localhost:5000/#/experiments/18
Images: {'cer': 0.2186, 'wer': 0.4042, 'empty_rate': 0.0242, 'avg_confidence': 0.8913} {'sample_p50_ms': 6.2, 'sample_p95_ms': 15.71, 'preprocess_p50_ms': 0.0, 'inference_p50_ms': 459.52, 'inference_p95_ms': 926.07, 'total_p50_ms': 467.79, 'total_p95_ms': 942.4, 'throughput_items_per_sec': 1.87, 'model_size_mb': 15.5, 'total_pipeline_time_sec': 66.662}


In [3]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_OCR) as exp:
    adapter = OcrExperimentAdapter(
        run_name="vid_rapidocr_cyrillic_uniform3",
        extractor=rapid_cyr,
        preprocessor=OcrPreprocessor(mode="raw"),
        sampler=KeyframeSampler(strategy="uniform_3"),
        records=vid_records,
        silver_dir=SILVER_DIR,
        rec_model_path=ROOT / CYRILLIC_OCR_MODEL_PATH,
        rec_keys_path=ROOT / CYRILLIC_OCR_DICT_PATH,
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "vid_uniform3", **res["quality"], **res["performance"]})
print("Video uniform_3:", res["quality"], res["performance"])

🏃 View run vid_rapidocr_cyrillic_uniform3 at: http://localhost:5000/#/experiments/18/runs/bb8e12bd6e2b4041ab53bfc52cfcfd90
🧪 View experiment at: http://localhost:5000/#/experiments/18
Video uniform_3: {'cer': 0.8726, 'wer': 0.9318, 'empty_rate': 0.4174, 'avg_confidence': 0.9184} {'sample_p50_ms': 230.84, 'sample_p95_ms': 505.81, 'preprocess_p50_ms': 0.0, 'inference_p50_ms': 1167.89, 'inference_p95_ms': 1955.38, 'total_p50_ms': 1377.17, 'total_p95_ms': 2260.38, 'throughput_items_per_sec': 0.7, 'model_size_mb': 15.5, 'total_pipeline_time_sec': 163.458}


In [4]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_OCR) as exp:
    adapter = OcrExperimentAdapter(
        run_name="vid_rapidocr_cyrillic_interval_dedup",
        extractor=rapid_cyr,
        preprocessor=OcrPreprocessor(mode="raw"),
        sampler=KeyframeSampler(strategy="interval_dedup"),
        records=vid_records,
        silver_dir=SILVER_DIR,
        rec_model_path=ROOT / CYRILLIC_OCR_MODEL_PATH,
        rec_keys_path=ROOT / CYRILLIC_OCR_DICT_PATH,
    )
    res = exp.create_pipeline(adapter).run()
results.append({"run": "vid_interval_dedup", **res["quality"], **res["performance"]})
print("Video interval_dedup:", res["quality"], res["performance"])
print(
    pd.DataFrame(results)[
        ["run", "cer", "wer", "empty_rate", "avg_confidence", "sample_p50_ms", "inference_p50_ms", "total_p50_ms"]
    ].to_string(index=False)
)

🏃 View run vid_rapidocr_cyrillic_interval_dedup at: http://localhost:5000/#/experiments/18/runs/57ef837aa7b24a659e5810c368d3ef30
🧪 View experiment at: http://localhost:5000/#/experiments/18
Video interval_dedup: {'cer': 0.8965, 'wer': 0.9426, 'empty_rate': 0.4609, 'avg_confidence': 0.9089} {'sample_p50_ms': 511.06, 'sample_p95_ms': 1018.95, 'preprocess_p50_ms': 0.0, 'inference_p50_ms': 850.94, 'inference_p95_ms': 2941.39, 'total_p50_ms': 1286.85, 'total_p95_ms': 4034.86, 'throughput_items_per_sec': 0.6, 'model_size_mb': 15.5, 'total_pipeline_time_sec': 191.353}
               run    cer    wer  empty_rate  avg_confidence  sample_p50_ms  inference_p50_ms  total_p50_ms
           img_raw 0.2186 0.4042      0.0242          0.8913           6.20            459.52        467.79
      vid_uniform3 0.8726 0.9318      0.4174          0.9184         230.84           1167.89       1377.17
vid_interval_dedup 0.8965 0.9426      0.4609          0.9089         511.06            850.94       1286.85
